In [35]:
import json
import torch
from rapidfuzz import fuzz
from itertools import combinations
from collections import Counter
from sentence_transformers import SentenceTransformer, util


In [36]:
# Load file 
with open("ontology_output/ontology_fabi_experimet.json", "r", encoding="utf-8") as f:
    data = json.load(f)

Count most common "Klassen" and "Eigenschaften"

In [37]:
class_counter = Counter()
property_counter = Counter()

for cluster in data:
    ontology = cluster.get("ontology", {})
    klassen = ontology.get("Klassen", [])
    eigenschaften = ontology.get("Eigenschaften", [])
    
    class_counter.update(klassen)
    property_counter.update(eigenschaften)

most_common_classes = class_counter.most_common(20)
most_common_properties = property_counter.most_common(20)

print(f"Most common classes: {most_common_classes}")
print(f"Most common properties: {most_common_properties}")

Most common classes: [('Rechtsgrundlage', 24), ('Rechtsfolge', 21), ('Vertrag', 15), ('Verjährung', 14), ('Anspruch', 13), ('Mangel', 13), ('Schadensersatzanspruch', 12), ('Schaden', 10), ('Mietvertrag', 9), ('Verpflichtung', 9), ('Klausel', 9), ('Kunde', 8), ('Haftung', 8), ('Mieter', 7), ('Käufer', 7), ('Verkäufer', 7), ('Kündigung', 7), ('Makler', 7), ('Vergütung', 7), ('Gesellschafter', 6)]
Most common properties: [('erfüllen', 30), ('gilt für', 23), ('beinhaltet', 22), ('betrifft', 21), ('verpflichtet', 21), ('erfüllt', 17), ('hat', 17), ('geltend machen', 17), ('gilt', 15), ('tritt ein', 14), ('tragen', 10), ('sicherstellen', 9), ('spielt eine Rolle', 9), ('spielen', 9), ('verletzen', 8), ('verletzt', 8), ('wirksam', 8), ('verlangen', 8), ('bestimmen', 8), ('basiert auf', 8)]


Calculate String similarities between classes and properites

Only implemented for claces, because some issues occured, because it found sting similarities in oposites, like in Verletzer <-> Verletzter, but it gives a good guidence for finding duplicates like in Netznutzungsentgelte <-> Netznutzungsentgelt.

In [38]:
all_classes = list(class_counter.keys())
similar_string_class_pairs = []

# Compare every pair
for a, b in combinations(all_classes, 2):
    score = fuzz.ratio(a, b)  # normalised Levenshtein similarity (0-100)
    if score >= 80:
        similar_string_class_pairs.append((a, b, score))

similar_string_class_pairs.sort(key=lambda x: -x[2])

for a, b, score in similar_string_class_pairs:
    print(f"{a} <-> {b} : {score}")

top_similar_classes_string = [
    {"class_1": a, "class_2": b, "levenshtein_score": score}
    for a, b, score in similar_string_class_pairs
]

with open("top_similar_classes_string.json", "w", encoding="utf-8") as f:
    json.dump(top_similar_classes_string, f, indent=2, ensure_ascii=False)

Rechtsfolge <-> Rechtsfolgen : 95.65217391304348
Versicherter <-> Versicherer : 95.65217391304348
Geschäftsunfähigkeit <-> Geschäftsfähigkeit : 94.73684210526316
Mangelbeseitigung <-> Mängelbeseitigung : 94.11764705882352
Beklagte <-> Beklagter : 94.11764705882352
Gemeinschuldnerin <-> Gemeinschuldner : 93.75
Gesellschafter <-> Gesellschafterin : 93.33333333333333
Schadensersatzanspruch <-> Schadensersatzansprüche : 93.33333333333333
Drittschuldnerin <-> Drittschuldner : 93.33333333333333
Verwaltungsrat <-> Verwaltungsakt : 92.85714285714286
Patent <-> Patient : 92.3076923076923
Gesellschaft <-> Gesellschafter : 92.3076923076923
Recht <-> Rechte : 90.9090909090909
Mieter <-> Miete : 90.9090909090909
Unternehmer <-> Unternehmen : 90.9090909090909
Vermieter <-> Vermieterin : 90.0
Bedingungen <-> Bedingung : 90.0
Vertragspartei <-> Vertragspartner : 89.65517241379311
Mängelansprüche <-> Mängelanspruch : 89.65517241379311
außerordentliche Kündigung <-> ordentliche Kündigung : 89.3617021276

Look for similarities with Sentince embeddings

In [39]:
model = SentenceTransformer('distiluse-base-multilingual-cased-v1')

class_names = list(class_counter.keys())

# Encode all class names to embeddings
embeddings = model.encode(class_names, convert_to_tensor=True)

similar_embedding_class_pairs = []

# Compare each pair
for i, j in combinations(range(len(class_names)), 2):
    sim_score = util.cos_sim(embeddings[i], embeddings[j]).item()
    if sim_score >= 0.8:  # Adjust threshold as needed
        similar_embedding_class_pairs.append((class_names[i], class_names[j], sim_score))

# Sort and display top results
similar_embedding_class_pairs.sort(key=lambda x: -x[2])

for a, b, score in similar_embedding_class_pairs[:20]:
    print(f"{a} <-> {b} : similarity = {score:.3f}")

top_similar_classes_embedding = [
    {"class_1": a, "class_2": b, "similarity": score}
    for a, b, score in similar_embedding_class_pairs
]

with open("top_similar_classes_embedding.json", "w", encoding="utf-8") as f:
    json.dump(top_similar_classes_embedding, f, indent=2, ensure_ascii=False)

Versicherter <-> Versicherer : similarity = 0.993
Gemeinschuldnerin <-> Gemeinschuldner : similarity = 0.983
Gesellschafter <-> Gesellschafterin : similarity = 0.982
Mängelrüge <-> Mängel : similarity = 0.977
Drittschuldnerin <-> Drittschuldner : similarity = 0.976
Abrechnung <-> Verrechnung : similarity = 0.975
Rechnung <-> Verrechnung : similarity = 0.973
Aufrechnung <-> Verrechnung : similarity = 0.972
Vermieter <-> Vermieterin : similarity = 0.971
Schadensabrechnung <-> Schadensverursachung : similarity = 0.967
Vertragsauslegung <-> Vertragsbestätigung : similarity = 0.967
Käufer <-> Verkäufer : similarity = 0.964
Rechnung <-> Aufrechnung : similarity = 0.964
Verschuldensgrad <-> Verschuldenshaftung : similarity = 0.964
Vertragsauslegung <-> Vertragserklärung : similarity = 0.963
Leasinggeber <-> Leasingnehmer : similarity = 0.962
Mängelrüge <-> Mängelanspruch : similarity = 0.962
Versicherter <-> Versicherung : similarity = 0.961
Vertragsbestätigung <-> Vertragsleistung : similari

In [40]:
duplicate_sets = [
    {"Kläger", "Klägerin"},
    {"Netznutzungsentgelt", "Netznutzungsentgelte"},
    {"Angeklagter", "Angeklagte"},
    {"Verurteilter", "Verurteilte"},
    {"Geräuschemission", "Geräuschemissionen"},
    {"Mangelbeseitigung", "Mängelbeseitigung", "Mangelbeseitigungen", "Mängelbeseitigungen"},
    {"Mangel", "Mängel"},
    {"Geschäft", "Geschäfte"},
    {"Beklagter", "Beklagte"},
    {"Geselschafter", "Gesellschafterin"},
    {"Voraussetzung", "Voraussetzungen"},
    {"Rechtsfolge", "Rechtsfolgen"},
    {"Beklagter", "Beklagte"},
    {"Gemeinschuldner", "Gemeinschuldnerin"},
    {"Schadensersatzanspruch", "Schadensersatzansprüche", "Schadensanspruch", "Schadensansprüche"},
    {"Drittschuldner", "Drittschuldnerin"},
    {"Recht", "Rechte"},
    {"Vermieter", "Vermieterin"},
    {"Bedingungen", "Bedingung"},
    {"Mangelanspruch", "Mangelansprüche", "Mängelanspruch", "Mängeransprüche"},
    {"Schadensersatzpflicht", "Schadensersatzverpflichtung"},
    {"Schadensersatz", "Schadensersatzleistung"},
    {"Gebühren", "Gebühr"},
    {"Geschäftsführer", "Geschäftsführerin", "Geschäftsführung"},
    {"Umstände", "Umstand"},
    {"Anklageschrift", "Klageschrift"}, # Eigentlich Zivil vs Strafprozess, aber ähnlich
    {"Richter", "Richteramt"}, # Vielleicht
    {"Schlichtungsverfahren", "Schlichtungsversuch"},
    {"Kündigungsklausel", "Hinauskündigungsklausel"},
    {"Verrechnung", "Aufrechnung"}


    

    


]